## Prescribe Velocity

In [ ]:
import json
import sys
from pathlib import Path
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "hamageolib").is_dir()
)
sys.path.insert(0, str(repo_root))

from hamageolib.utils.aspect.io import save_parameters_from_dict

## Create case

### Create a Stegman Case

In [ ]:
is_make_stegman_case = False

if is_make_stegman_case:

    from hamageolib.research.prescribe_vecloity.config import make_stegman_case
    
    # 2-D case
    case_dir = Path("/mnt/lochy/ASPECT_DATA/MOW/test_prescribe")/"stegman_2d_prescribed_velocity_0"
    case_dir.mkdir(parents=True, exist_ok=False)
    
    prm_dict, wb_dict, context = make_stegman_case({
          "end_time": 5e6,
          "adaptive_refinement": 2,
          "refine_plate_with_isosurfaces": True,
          "long_slab": True, # longer slab by 300 km length
          "prescribe_slab_velocity": True, # prescribed condition
          "slab_velocity_magnitude": 0.05 # prescribed condition, m/year
    })
    
    # 3-D case
    # case_dir = Path("/mnt/lochy/ASPECT_DATA/MOW/test_prescribe")/"stegman_3d_1"
    # case_dir.mkdir(parents=True, exist_ok=True)
    # prm_dict, wb_dict, context = make_stegman_case({
    #     "dimension": 3,
    #       "x_repetitions": 4,
    #       "y_repetitions": 2,
    #       "z_repetitions": 1,
    #       "end_time": 5e6,
    #       "graphical_output_interval": 5e5,
    #       "checkpoint_interval": 5e5,
    #       "adaptive_refinement": 2,
    #       "refine_plate_with_isosurfaces": True,
    #       "trench_width": 1200e3, # slab width
    #       "long_slab": True, # longer slab by 300 km length
    #       "prescribe_slab_velocity": True, # prescribed condition
    #       "slab_velocity_magnitude": 0.05, # prescribed condition, m/year
    # })
    
    
    with (case_dir / "case.prm").open("w") as prm_file:
        save_parameters_from_dict(prm_file, prm_dict)
    
    with (case_dir / "case.wb").open("w") as wb_file:
        json.dump(wb_dict, wb_file, indent=2)
        wb_file.write("\n")
    
    print("New case: %s" % case_dir)

### Modify case domain and position

Take an existing case and change its chunk longitude bounds and/or rotate its complete WorldBuilder geometry along longitude. This step does **not** retreat the trench, fit the slab, change segment definitions, or add overlap features. The current settings retain `_2 → _5`, longitude bounds −5° to 35°, and zero longitude shift. Choose a fresh destination before enabling generation.

`None` bounds inherit the source; `longitude_shift_deg=5.0` shifts the complete slab 5° east, including dip points, cross section and global diagnostics. Radii, latitude, material/solver settings and mesh repetitions remain inherited. Widening the domain with unchanged repetitions coarsens its base angular spacing. Longitudes are not wrapped at the dateline.

Original four-segment cases can be sampled directly. Already-retreated or sectioned cases are also supported when their matching `geometry.npz` is present; those diagnostics are rotated without refitting. Bounds changes use gdmate's standard parser/writer, preserving parameter values but not comments/formatting. Unchanged input files are copied byte-for-byte. New inputs, geometry diagnostics and `domain_report.json` are written; simulation outputs are not copied and ASPECT is not run. Existing destinations are refused.


In [ ]:
from pathlib import Path

# Enable only after choosing a new destination directory.
do_modify_case_domain = True

domain_source_case = Path(
    "/mnt/lochy/ASPECT_DATA/MOW/test_prescribe/"
    "3d_spherical_subduction_along_surface_velocity_2"
)
domain_destination_case = domain_source_case.parent / (
    "3d_spherical_subduction_along_surface_velocity_5"
)

# Optional chunk bounds in degrees; None inherits the source (currently 0..25).
chunk_min_longitude = -5.0  # e.g. -5.0
chunk_max_longitude = 35.0  # e.g. 35.0 to enlarge the eastern side
# Rotate the whole slab and its anchored reference together about Earth's axis.
longitude_shift_deg = 0.0   # e.g. +5.0; enlarge the bounds to contain the deep slab
# Longitude mesh repetitions remain inherited from the source case.

if do_modify_case_domain:
    import json
    import sys

    domain_repo_root = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "hamageolib").is_dir()
    )
    if str(domain_repo_root) not in sys.path:
        sys.path.insert(0, str(domain_repo_root))

    from hamageolib.research.prescribe_vecloity.case_domain import modify_case_domain

    domain_report = modify_case_domain(
        domain_source_case,
        domain_destination_case,
        chunk_min_longitude=chunk_min_longitude,
        chunk_max_longitude=chunk_max_longitude,
        longitude_shift_deg=longitude_shift_deg,
    )
    print(json.dumps(domain_report, indent=2))
    print(f"Created case: {domain_destination_case}")


### Generate a trench-retreat case

Take an existing original four-segment slab case (including a case whose domain/position was changed above) and retreat its trench while retaining its deep terminal geometry. This block defaults to `_5 → _6` and a 100 km retreat, and is disabled until you choose to run it. Both blocks have their own paths and can run independently; the domain case must exist before using it as the retreat source.

Only source/destination paths and `trench_retreat_m` are configured here. Domain bounds and longitude placement are inherited unchanged from the source. `None` keeps the original slab; a positive value in metres fits the retreat geometry with 20 km overlap collars. The standard `_2` profile at 100 km reproduces the `_3` benchmark. The complete `generate_retreat_case` API still accepts combined domain/shift options for backwards compatibility, but they are not used by this block.

Sources must retain the supported east-dipping four-segment profile with constant thickness and equal transition lengths; an already-retreated two-feature case cannot be retreated recursively. Use the domain block to move such a case without changing its slab shape. The utility writes `case.prm`, `case.wb`, `geometry.npz`, and `geometry_report.json`; it refuses existing destinations and never runs ASPECT. New retreat distances require geometry/field/mesh and solver validation before production use.


In [ ]:
from pathlib import Path

# Independent of the domain-modification block: choose existing input and new output.
do_generate_retreat_case = False
retreat_source_case = Path(
    "/mnt/lochy/ASPECT_DATA/MOW/test_prescribe/"
    "3d_spherical_subduction_along_surface_velocity_5"
)
retreat_destination_case = retreat_source_case.parent / (
    "3d_spherical_subduction_along_surface_velocity_6"
)
trench_retreat_m = 100e3  # Metres; None keeps the original slab unchanged.

if do_generate_retreat_case:
    import json
    import sys

    retreat_repo_root = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "hamageolib").is_dir()
    )
    if str(retreat_repo_root) not in sys.path:
        sys.path.insert(0, str(retreat_repo_root))
    from hamageolib.research.prescribe_vecloity.trench_retreat import generate_retreat_case

    retreat_report = generate_retreat_case(
        retreat_source_case,
        retreat_destination_case,
        trench_retreat_m=trench_retreat_m,
    )
    print(json.dumps(retreat_report, indent=2))
    print(f"Created case: {retreat_destination_case}")


## Prescribed-velocity visualization

view: green translucent plate (0.8–1.0), white domain velocity arrows, and blue slab velocity arrows. Edit the case, saved time, glyph scale and sample counts below, then enable the block. The image and a reusable ParaView state are saved under `img/pv_outputs`.

The runner loads `base.py` and the separate prescribed-velocity workflow at runtime; it does not concatenate or modify either source. `PVPYTHON` can select a ParaView Python executable. Run this block independently of the case-creation cell above.


In [ ]:
from pathlib import Path

do_prescribed_velocity_visualization = True
visualization_case_dir = Path(
    "/mnt/lochy/ASPECT_DATA/MOW/test_prescribe/"
    "3d_spherical_subduction_along_surface_velocity_5"
)
visualization_time = 0.0  # Years; choose a saved PVD timestep.
velocity_glyph_scale = 5e6
velocity_domain_samples = 5000
velocity_slab_samples = 500

if do_prescribed_velocity_visualization:
    import os
    import shutil
    import subprocess
    from pathlib import Path

    visual_repo_root = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "hamageolib").is_dir()
    )
    visual_scripts = visual_repo_root / "scripts/paraview_scripts"
    solution_file = visualization_case_dir / "output/solution.pvd"
    if not solution_file.is_file():
        raise FileNotFoundError(solution_file)
    pvpython_executable = shutil.which(os.environ.get("PVPYTHON", "pvpython"))
    if pvpython_executable is None:
        raise FileNotFoundError("Set PVPYTHON to a ParaView pvpython executable")

    image_dir = visualization_case_dir / "img/pv_outputs"
    script_dir = visualization_case_dir / "paraview_scripts"
    image_dir.mkdir(parents=True, exist_ok=True)
    script_dir.mkdir(parents=True, exist_ok=True)
    image_file = image_dir / "prescribed_velocity.png"
    state_file = image_dir / "prescribed_velocity.pvsm"
    runner_file = script_dir / "run_prescribed_velocity.py"
    runner_lines = [
        "import os, runpy",
        "os.environ.setdefault('PV_INTERFACE_PATH', '')",
        f"base = runpy.run_path({str(visual_scripts / 'base.py')!r})",
        f"workflow = runpy.run_path({str(visual_scripts / 'PrescribedVelocity/prescribed_velocity.py')!r}, init_globals=base)",
        "workflow['plot_prescribed_velocity'](",
        f"    {str(solution_file)!r}, {str(image_file)!r},",
        f"    time={visualization_time!r}, glyph_scale={velocity_glyph_scale!r},",
        f"    domain_samples={velocity_domain_samples!r}, slab_samples={velocity_slab_samples!r},",
        f"    state_file={str(state_file)!r})",
    ]
    runner_file.write_text("\n".join(runner_lines) + "\n", encoding="utf-8")

    print("Saved paraview script: %s" % str(runner_file))
    # subprocess.run(
    #     [pvpython_executable, "--force-offscreen-rendering", str(runner_file)],
    #     check=True,
    # )
    # print(f"Saved image: {image_file}")
    # print(f"Saved ParaView state: {state_file}")
